# Bài 16/25 · LeetCode #100 — Same Tree

**Mức độ:** Easy · **Chủ đề:** Binary Tree · Recursion · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/same-tree/](https://leetcode.com/problems/same-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho hai gốc cây nhị phân `p`, `q`. Trả `True` nếu hai cây có **cùng hình dạng** và giá trị ở mỗi vị trí tương ứng **bằng nhau**; ngược lại trả `False`. Hai cây cùng thứ tự giá trị nhưng khác nhánh vẫn là khác nhau.

### Ví dụ quan trọng

`p=[1,2,3], q=[1,2,3] → True`. `p=[1,2], q=[1,None,2] → False` vì node `2` nằm ở nhánh khác. `p=None,q=None → True`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [1]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. `TreeNode` thật sự là gì? Học từ một nút đơn

**Binary Tree (cây nhị phân)** gồm các nút (*node*). Mỗi nút lưu đúng ba thông tin:
`val` (giá trị), `left` (tham chiếu đến con trái), `right` (tham chiếu đến con phải).

Quan trọng: `left`/`right` **không phải** là vị trí trong mảng; đó là liên kết đến một **đối tượng** `TreeNode` khác.
`None` nghĩa là *không có nút*. Một cây rỗng được biểu diễn bởi `root = None`.

Ví dụ cây `1` có con trái `2` và con phải `3`:

```text
      1
     / \
    2   3
```

Nhìn vào code dựng cây dưới đây, hãy dự đoán từng phép in trước khi chạy.

In [2]:
root_demo = TreeNode(1)
root_demo.left = TreeNode(2)
root_demo.right = TreeNode(3)
print("Gốc:", root_demo.val)
print("Con trái:", root_demo.left.val)
print("Con phải:", root_demo.right.val)
print("Con trái của nút 2:", root_demo.left.left)  # None

Gốc: 1
Con trái: 2
Con phải: 3
Con trái của nút 2: None


### 2.2. Tham chiếu (*reference*) và giá trị khác nhau thế nào?

`a = root_demo.left` **không sao chép** một nút mới: `a` và `root_demo.left` là hai tên cùng chỉ đến một đối tượng.
Vì vậy `a.val = 99` sẽ thay đổi giá trị trong cây gốc.

- `a is b`: hỏi **có cùng một đối tượng** hay không.
- `a.val == b.val`: hỏi **hai giá trị bằng nhau** hay không.
- Hai nút khác nhau có thể cùng giá trị; với so sánh cây, ta cần xét cả cấu trúc, không chỉ xét một con số.

Phần này đặc biệt quan trọng khi gắn nút con trong giải pháp chính.

In [3]:
x_demo = TreeNode(7)
y_demo = TreeNode(7)
z_demo = x_demo
print("Giá trị bằng nhau:", x_demo.val == y_demo.val)  # True
print("Là một đối tượng:", x_demo is y_demo)        # False
print("Chỉ đến cùng nút:", x_demo is z_demo)       # True
z_demo.val = 8
print("Giá trị của x sau khi đổi qua z:", x_demo.val)

Giá trị bằng nhau: True
Là một đối tượng: False
Chỉ đến cùng nút: True
Giá trị của x sau khi đổi qua z: 8


### 2.3. `None`, `is None` và thứ tự kiểm tra an toàn

Với `node = None`, viết `node.val` sẽ gây `AttributeError`: không thể truy cập `val` của vật không tồn tại.
Cách an toàn: kiểm tra `node is None` **trước**, rồi mới dùng `node.val`.

Trong Python, biểu thức `A and B` có **short-circuit** (đoản mạch): nếu `A` là `False` thì Python **không đánh giá** `B` nữa.
Tương tự, với `A or B`, nếu `A` là `True` thì không cần tính `B`.
Đây là lý do ta có thể xử lý trường hợp rỗng trước khi truy cập thuộc tính và tránh nhánh đệ quy vô ích.

In [4]:
nothing = None
print("Là nút rỗng?", nothing is None)
print("Không phải nút rỗng?", nothing is not None)
print("Đọc an toàn:", nothing.val if nothing is not None else "không có nút")
leaf_demo = TreeNode(42)
print("Lá có hai con rỗng:", leaf_demo.left is None and leaf_demo.right is None)

Là nút rỗng? True
Không phải nút rỗng? False
Đọc an toàn: không có nút
Lá có hai con rỗng: True


### 2.4. Hiểu danh sách level-order và vì sao phải giữ `None`

LeetCode thường hiển thị cây theo thứ tự từng tầng, trái trước phải sau (*level-order*).
Trong notebook, hàm `tree(values)` dựng cây từ dạng ấy để dễ thử, còn `show_tree(root)` đưa cây về danh sách để đọc.
**Đừng coi list đó là định nghĩa của cây**: thuật toán nộp LeetCode nhận `TreeNode`, không nhận danh sách.

Ví dụ `[1,2,None,3]` là: gốc 1 có con trái 2, nút 2 có con trái 3.
Nó khác `[1,None,2,3]`: gốc 1 có con phải 2, nút 2 có con trái 3.
Dù phần tử `3` xuất hiện cuối cả hai danh sách, vị trí con trỏ khác nhau hoàn toàn.

In [5]:
shape_a = tree([1, 2, None, 3])
shape_b = tree([1, None, 2, 3])
print("A:", show_tree(shape_a), "gốc trái:", shape_a.left)
print("B:", show_tree(shape_b), "gốc trái:", shape_b.left)
print("A.gốc.phải:", shape_a.right, "B.gốc.phải:", shape_b.right)
assert show_tree(shape_a) != show_tree(shape_b)

A: [1, 2, None, 3] gốc trái: TreeNode(2)
B: [1, None, 2, 3] gốc trái: None
A.gốc.phải: None B.gốc.phải: TreeNode(2)


### 2.5. Đệ quy (*recursion*) từ con số 0: hàm gọi lại chính nó

Đệ quy có hai mảnh **bắt buộc**:

1. **Base case** (trường hợp dừng): đơn giản đến mức trả lời ngay, không gọi tiếp.
2. **Recursive step** (bước rút gọn): gọi lại hàm trên bài toán nhỏ hơn, rồi dùng kết quả ấy.

**Call stack (ngăn xếp lời gọi)** giữ lại vị trí mà mỗi lần gọi đang chờ `return`.
Ví dụ `countdown(3)` cần đi `3 → 2 → 1 → 0` rồi quay trở ra.
Trên cây, bước nhỏ hơn tự nhiên là cây con `node.left` hoặc `node.right`.

Kí hiệu chiều cao `h`: một đường từ gốc xuống lá dài tối đa `h` nút, nên tầng gọi lồng nhau có thể đến `h`.
Đệ quy không tự biến mất bộ nhớ: mỗi lần gọi đang chờ chiếm một phần call stack.

In [6]:
def countdown_demo(n, level=0):
    print("  " * level + f"vào n={n}")
    if n == 0:
        print("  " * level + "đã đến base case")
        return 0
    result = 1 + countdown_demo(n - 1, level + 1)
    print("  " * level + f"trở lại n={n}, trả {result}")
    return result

assert countdown_demo(3) == 3

vào n=3
  vào n=2
    vào n=1
      vào n=0
      đã đến base case
    trở lại n=1, trả 1
  trở lại n=2, trả 2
trở lại n=3, trả 3


### 2.6. `Optional[TreeNode]` và `return` là gì?

`Optional[TreeNode]` (trong `typing`) mô tả kiểu dữ liệu có thể là một `TreeNode` **hoặc** `None`.
Đây là **type hint** (gợi ý kiểu), không làm Python tự kiểm tra cây hợp lệ trong lúc chạy.

`return` *trả kết quả về nơi gọi và kết thúc lượt chạy hiện tại*. `print` chỉ đưa thông tin lên màn hình;
một hàm chỉ `print(True)` mà không `return True` sẽ trả về `None` (không đúng kiểu `bool` mà nhiều bài đòi).

Vòng lặp `for`, phép `max`, phép `abs`, danh sách `[]`, biểu thức `x is None`, phép truy cập `.left` / `.right`
đều sẽ được giải thích lại ngay tại chỗ cần dùng ở phần sau.

In [7]:
def check_node_demo(node: Optional[TreeNode]) -> bool:
    if node is None:
        return False
    return node.val > 0

print("Kiểm tra None:", check_node_demo(None))
print("Kiểm tra nút 5:", check_node_demo(TreeNode(5)))
print("Kiểm tra nút -2:", check_node_demo(TreeNode(-2)))

Kiểm tra None: False
Kiểm tra nút 5: True
Kiểm tra nút -2: False


## 3. Python nền tảng — công cụ cần hiểu trước khi giải

**Tham chiếu node**: `node.left` chứa một `TreeNode` hoặc `None`. Toán tử `is None` kiểm tra nhánh trống, `and` yêu cầu **cả hai** điều kiện đúng.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [8]:
a=tree([1,2,3]); b=tree([1,None,2])
print("a:",show_tree(a),"b:",show_tree(b))
print("a.left =",a.left,"b.left =",b.left)
print("a.left is None?",a.left is None)
print("Hai gốc có cùng giá trị?",a.val == b.val)

a: [1, 2, 3] b: [1, None, 2]
a.left = TreeNode(2) b.left = None
a.left is None? False
Hai gốc có cùng giá trị? True


### 3.1. Một nút khác giá trị, một cây khác hình dáng

Đừng chỉ hỏi “các số có giống nhau không?”. Hai cây còn phải đồng ý ở **mọi vị trí** con trái, con phải.
Cặp `A=[1,2,3]`, `B=[1,3,2]` có cùng tập giá trị `{1,2,3}` nhưng **không** cùng cây.
Cặp `A=[1,2]`, `B=[1,None,2]` có cùng hai giá trị `[1,2]` nhưng một nút nằm trái, một nút nằm phải.

Đọc kỹ ba cặp ví dụ. Nếu câu trả lời của bạn đều dựa vào `sorted(values)`, thuật toán sẽ trả sai.
Để so sánh **cấu trúc**, ta cần ghép hai nút ở **cùng một vị trí** rồi so sánh đồng thời.

In [9]:
pairs_demo = [
    ([1, 2, 3], [1, 3, 2]),
    ([1, 2], [1, None, 2]),
    ([1, 2, 3], [1, 2, 3]),
]
for a, b in pairs_demo:
    print("A=", a, " B=", b, " -> kết luận đúng:", a == b)

A= [1, 2, 3]  B= [1, 3, 2]  -> kết luận đúng: False
A= [1, 2]  B= [1, None, 2]  -> kết luận đúng: False
A= [1, 2, 3]  B= [1, 2, 3]  -> kết luận đúng: True


### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Ta có thể chuyển từng cây thành dãy cặp `(vị trí, giá trị)`, **bao gồm cả dấu None**, rồi so sánh hai dãy. Cách này dễ quan sát nhưng cần tạo danh sách phụ O(n).

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [10]:
def baseline(p,q):
    def encode(root):
        if root is None: return '#'
        return (root.val,encode(root.left),encode(root.right))
    return encode(p)==encode(q)

In [11]:
for a,b in [([1,2,3],[1,2,3]),([1,2],[1,None,2]),([],[])]:
    print(a,b,"->",baseline(tree(a),tree(b)))

[1, 2, 3] [1, 2, 3] -> True
[1, 2] [1, None, 2] -> False
[] [] -> True


### 4.1. Mổ xẻ cách baseline: “mã hóa toàn bộ cây”

Trong `baseline` phía trên, `encode(node)` biến cây thành các bộ ba:
`(node.val, encode(node.left), encode(node.right))`. `None` được mã hóa thành dấu `'#'`.
**Tuple (bộ)** là dữ liệu dạng `(a, b, c)`, so sánh `==` theo từng thành phần theo thứ tự.
Ví dụ `('1', '#', '#')` mô tả cả hai vị trí con của nút, nên không làm thất lạc hình dáng.

**Nhược điểm**: ta tạo ra cấu trúc trung gian của toàn bộ hai cây rồi mới so sánh.
Khi gặp một khác biệt ở gốc, ta đã làm thừa công việc mã hóa tất cả hậu duệ.
Ý tưởng cải tiến: **so sánh ngay khi đi xuống**; thấy khác thì `return False` ngay.

In [12]:
def encoded_shape_demo(node):
    if node is None:
        return "#"
    return (node.val, encoded_shape_demo(node.left), encoded_shape_demo(node.right))

a = tree([1, 2, 3])
b = tree([1, None, 2, 3])
print("Mã hóa A:", encoded_shape_demo(a))
print("Mã hóa B:", encoded_shape_demo(b))
print("Bằng nhau?", encoded_shape_demo(a) == encoded_shape_demo(b))

Mã hóa A: (1, (2, '#', '#'), (3, '#', '#'))
Mã hóa B: (1, '#', (2, (3, '#', '#'), '#'))
Bằng nhau? False


### 4.2. Theo dõi số lần ghé nút: vì sao kiểm tra sớm đáng giá?

Nếu một cây có hàng nghìn nút nhưng gốc khác nhau, cách mã hóa phải tạo ra mọi bộ ba của cả hai cây.
Cách so sánh trực tiếp nhìn **hai nút gốc** rồi dừng.
Thời gian trong trường hợp xấu nhất của cả hai vẫn `O(n + m)` với `n, m` là số nút hai cây,
nhưng khả năng **dừng sớm** mang lại lợi ích thực tế và không phải tạo cả chuỗi/bộ biểu diễn.

Ta thử ghi số lượt thăm trong một mô hình minh họa, không đo thời gian bằng đồng hồ vì tốc độ phụ thuộc máy.

In [13]:
def count_direct_comparisons(a, b):
    visited = 0
    def go(x, y):
        nonlocal visited
        visited += 1  # đếm một cặp vị trí được kiểm tra
        if x is None or y is None:
            return x is y
        if x.val != y.val:
            return False
        return go(x.left, y.left) and go(x.right, y.right)
    return go(a, b), visited

long_a = tree([0, 1, 2, 3, 4, 5, 6])
long_b = tree([99, 1, 2, 3, 4, 5, 6])
print("Khác ngay tại gốc:", count_direct_comparisons(long_a, long_b))
print("Giống nhau toàn bộ:", count_direct_comparisons(long_a, tree([0, 1, 2, 3, 4, 5, 6])))

Khác ngay tại gốc: (False, 1)
Giống nhau toàn bộ: (True, 15)


## 5. Vì sao cần chuyển sang kỹ thuật chính?

Thay vì lưu cả cây dưới dạng chuỗi/tuple, ta so sánh **ngay khi gặp hai node**. Trường hợp dừng là `None`: cả hai None → giống; chỉ một None → khác.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Đệ quy theo cặp `(a,b)`: (1) cùng rỗng thì `True`; (2) một rỗng hoặc giá trị khác thì `False`; (3) so sánh cây con trái **và** cây con phải.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

In [14]:
def solve(p: Optional[TreeNode], q: Optional[TreeNode]) -> bool:
    def same(a,b):
        if a is None and b is None: return True
        if a is None or b is None: return False
        if a.val != b.val: return False
        return same(a.left,b.left) and same(a.right,b.right)
    return same(p,q)

### 6.1. Từ ba trường hợp dừng đến code hoàn chỉnh

Hãy gọi `same(a,b)` là câu hỏi “hai cây con có gốc `a`,`b` giống hệt nhau?”.

**Trường hợp 1:** cả `a` và `b` là `None`: cả hai nhánh đều trống ⇒ `True`.

**Trường hợp 2:** chỉ một bên là `None`: một bên thiếu nút ⇒ `False`.

**Trường hợp 3:** cả hai có nút, nhưng `a.val != b.val` ⇒ `False`.

**Nếu qua ba cửa đó**: hai giá trị gốc đã khớp, chỉ còn phải kiểm tra `same(a.left,b.left)` **và**
`same(a.right,b.right)`; `and` đòi cả hai câu đều đúng.

Điều kiện dừng luôn xuất hiện **trước** khi truy cập `.val`: đó là cách phòng `AttributeError`.

Đây là đệ quy theo **cặp cây con**, không phải duyệt một cây rồi “đoán” cây còn lại.

In [15]:
# Thí nghiệm rất nhỏ: chỉ hai nút lá cùng giá trị
left_leaf = TreeNode(8)
right_leaf = TreeNode(8)
print("Hai lá bằng nhau:", solve(left_leaf, right_leaf))
right_leaf.val = 9
print("Đổi đúng một giá trị:", solve(left_leaf, right_leaf))
print("Bên trái là None, bên phải có nút:", solve(None, right_leaf))
print("Cả hai đều None:", solve(None, None))

Hai lá bằng nhau: True
Đổi đúng một giá trị: False
Bên trái là None, bên phải có nút: False
Cả hai đều None: True


## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Cặp node | Kiểm tra | Quyết định |
|---|---|---|
| `1,1` | giá trị bằng nhau | xét trái và phải |
| `2,2` | bằng nhau | tiếp tục |
| `None,None` | cả hai trống | trả True |
| `3,4` | khác nhau | trả False → cả cây False |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [16]:
def trace_same(a,b,depth=0):
    pre='  '*depth
    print(pre,'so:',None if a is None else a.val,None if b is None else b.val)
    if a is None or b is None:
        result=a is None and b is None
    elif a.val!=b.val:
        result=False
    else:
        result=trace_same(a.left,b.left,depth+1) and trace_same(a.right,b.right,depth+1)
    print(pre,'=>',result)
    return result
trace_same(tree([1,2,3]),tree([1,2,4]))

 so: 1 1
   so: 2 2
     so: None None
     => True
     so: None None
     => True
   => True
   so: 3 4
   => False
 => False


False

### 7.1. Bảng trace bằng tay: hai cây chỉ khác ở nhánh phải

Xét `p=[1,2,3]` và `q=[1,2,4]`. Hàm so sánh *trái trước, phải sau*.

| Thứ tự | Lời gọi | Nhánh điều kiện | Kết quả |
|---:|---|---|---|
| 1 | `same(1,1)` | Gốc bằng nhau → hỏi trái, phải | Chờ |
| 2 | `same(2,2)` | Giá trị 2 bằng nhau | Chờ |
| 3 | `same(None,None)` (con trái của 2) | Cả hai trống | `True` |
| 4 | `same(None,None)` (con phải của 2) | Cả hai trống | `True` |
| 5 | Trở lại `same(2,2)` | `True and True` | `True` |
| 6 | `same(3,4)` | 3 khác 4 | `False` |
| 7 | Trở lại `same(1,1)` | `True and False` | **`False`** |

**Bất biến đọc trace:** mỗi cặp đang kiểm tra đều ở cùng đường đi `L/R` từ gốc.
Đây là điều ngăn thuật toán so nhầm nút “cùng giá trị nhưng khác vị trí”.

### 7.2. Biến thể không đệ quy: dùng stack (ngăn xếp) tường minh

Stack là danh sách mà ta **thêm ở cuối** bằng `.append(x)` và **lấy phần tử cuối** bằng `.pop()`.
Nó hoạt động theo **LIFO** (*Last In, First Out*: vào sau ra trước).

Ta lưu cặp `(nút cây 1, nút cây 2)` vào stack. Lấy một cặp ra, so như ba trường hợp dừng,
rồi đẩy **cặp con phải** và **cặp con trái** vào stack. Vì LIFO, cặp con trái sẽ được lấy trước.

Cách này hữu ích nếu bạn muốn thấy rõ trạng thái duyệt hoặc tránh giới hạn đệ quy Python với cây rất sâu.
Dù sử dụng stack, số lượng cặp cần xét vẫn tuyến tính theo số nút trong trường hợp xấu nhất.

In [17]:
def same_tree_iterative(p, q):
    stack = [(p, q)]
    while stack:
        a, b = stack.pop()
        if a is None and b is None:
            continue
        if a is None or b is None:
            return False
        if a.val != b.val:
            return False
        stack.append((a.right, b.right))
        stack.append((a.left, b.left))
    return True

for x, y in [([1, 2, 3], [1, 2, 3]), ([1, 2], [1, None, 2])]:
    print(x, y, "->", same_tree_iterative(tree(x), tree(y)))

[1, 2, 3] [1, 2, 3] -> True
[1, 2] [1, None, 2] -> False


### 7.3. Trace stack: trạng thái trước và sau thao tác

Ở ví dụ dưới, `stack` chứa **các cặp nút**; để dễ nhìn, ta chỉ in `val` hoặc `None`.
Hãy quan sát cặp con trái và con phải được thêm **sau khi** xác nhận hai nút gốc hợp lệ.
Khác với so sánh một danh sách giá trị, cặp `(a.left,b.left)` luôn giữ quan hệ “cùng hướng”.

In [18]:
def trace_same_tree_stack(p, q):
    stack = [(p, q)]
    step = 0
    while stack:
        step += 1
        a, b = stack.pop()
        label = lambda n: None if n is None else n.val
        print(f"B{step}: pop=({label(a)}, {label(b)}) | còn {len(stack)} cặp")
        if a is None and b is None:
            print("  Hai vị trí trống: tiếp tục")
            continue
        if a is None or b is None or a.val != b.val:
            print("  KHÁC -> False")
            return False
        stack.append((a.right, b.right))
        stack.append((a.left, b.left))
        print("  Khớp, đẩy con phải rồi con trái")
    print("Không có cặp sai: True")
    return True

assert not trace_same_tree_stack(tree([1, 2, 3]), tree([1, 2, 4]))

B1: pop=(1, 1) | còn 0 cặp
  Khớp, đẩy con phải rồi con trái
B2: pop=(2, 2) | còn 1 cặp
  Khớp, đẩy con phải rồi con trái
B3: pop=(None, None) | còn 2 cặp
  Hai vị trí trống: tiếp tục
B4: pop=(None, None) | còn 1 cặp
  Hai vị trí trống: tiếp tục
B5: pop=(3, 4) | còn 0 cặp
  KHÁC -> False


## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Mỗi lần gọi hàm xử lý cùng một **vị trí** trên hai cây. Cây con rỗng đúng khi cả hai rỗng. Hai node không rỗng chỉ có thể giống nếu giá trị gốc giống **và** hai cặp cây con tương ứng giống. Quy nạp theo số node: điều kiện trên đủ và cần để toàn bộ hai cây giống nhau.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [19]:
from typing import Optional
class Solution:
    def isSameTree(self, p: Optional[TreeNode], q: Optional[TreeNode]):
        def same(a,b):
            if a is None and b is None: return True
            if a is None or b is None: return False
            if a.val != b.val: return False
            return same(a.left,b.left) and same(a.right,b.right)
        return same(p,q)

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [20]:
s=Solution()
for p,q,expected in [([1,2,3],[1,2,3],True),([1,2],[1,None,2],False),([1,2,1],[1,1,2],False),([],[],True),([], [1],False)]:
    assert solve(tree(p),tree(q))==expected
    assert s.isSameTree(tree(p),tree(q))==expected
print('PASS 5 ca Same Tree')

PASS 5 ca Same Tree


### 10.1. Kiểm thử nhiều cây: so cả ba cách và kiểm tra tính đối xứng của quan hệ “bằng nhau”

Cần thử ngoài ví dụ đề: rỗng/rỗng, rỗng/có nút, trùng giá trị nhưng vị trí khác,
âm, cây lệch trái/phải, giá trị rất lớn, nút lá đơn lẻ.

Ba cách (`baseline`, `solve`, `same_tree_iterative`) phải cho cùng kết quả.
Ngoài ra `same(p,q) == same(q,p)` luôn đúng: quan hệ “giống hệt” không phụ thuộc thứ tự hai cây.

In [21]:
test_trees = [
    [], [1], [1, 2], [1, None, 2], [1, 2, 3],
    [1, 2, 3, 4, None, None, 5], [1, 2, 3, None, 4],
    [-5, -7, -3], [10, 10, 10, 10, 10]
]
checks = 0
for left in test_trees:
    for right in test_trees:
        a, b = tree(left), tree(right)
        want = baseline(a, b)
        assert solve(a, b) == want
        assert same_tree_iterative(a, b) == want
        assert solve(a, b) == solve(b, a)
        checks += 1
print(f"PASS {checks} cặp cây × 3 phép đối chiếu")

PASS 81 cặp cây × 3 phép đối chiếu


### 10.2. Trường hợp dễ lọt: cùng thứ tự duyệt inorder nhưng cây khác

*Inorder traversal* duyệt “con trái → nút → con phải”. Nó **không đủ** để định danh một cây.
Cây có một nút `2` nằm bên trái gốc `1` và cây có `2` nằm bên phải gốc `1` vẫn có thể trùng tập giá trị,
nhưng cấu trúc khác; so chỉ các giá trị mà bỏ `None` chắc chắn có thể trả sai.

Với Same Tree, cách an toàn là kiểm tra **nút ở chính xác vị trí tương ứng**, hoặc mã hóa có dấu `None`.

In [22]:
one_side_left = tree([1, 2])
one_side_right = tree([1, None, 2])
print("Cấu trúc A:", show_tree(one_side_left))
print("Cấu trúc B:", show_tree(one_side_right))
print("Giá trị xuất hiện ở cả hai:", [1, 2])
assert not solve(one_side_left, one_side_right)

Cấu trúc A: [1, 2]
Cấu trúc B: [1, None, 2]
Giá trị xuất hiện ở cả hai: [1, 2]


## 11. Phân tích Time/Space Complexity

Đặt `n` là tổng số node của hai cây. **Time O(n)** trong trường hợp phải duyệt hết. **Space O(h)** cho ngăn xếp đệ quy (`h` là độ sâu lớn nhất, tối đa O(n)); cách baseline tạo biểu diễn O(n) thêm.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Chỉ so sánh `a.val==b.val`: quên cấu trúc nhánh.
- ❌ `a is None or b is None: return True`: phải phân biệt **cả hai** hay **một** None.
- ❌ Dùng `or` để kết hợp so sánh trái/phải: phải dùng `and`.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Hai đoạn code sai phổ biến — đọc lỗi rồi tự sửa

**Lỗi A — thiếu trường hợp một bên `None`:** code gọi `.val` ngay cả khi `a` hoặc `b` rỗng.
Một bên có cây, một bên rỗng sẽ văng lỗi thay vì trả `False`.

**Lỗi B — dùng `or` thay vì `and`:** hai nhánh chỉ cần *một* bên giống là báo `True`.
Nhưng Same Tree yêu cầu **cả trái và phải cùng giống**.

Để sửa, xét `None` trước, so giá trị gốc, rồi ghép hai mệnh đề con bằng `and`.
Cell dưới mô phỏng **lỗi B** một cách an toàn, để thấy phản ví dụ cụ thể.

In [23]:
def wrong_same_uses_or(a, b):
    if a is None or b is None:
        return a is b
    return a.val == b.val and (
        wrong_same_uses_or(a.left, b.left)
        or wrong_same_uses_or(a.right, b.right)
    )

x, y = tree([1, 2, 3]), tree([1, 2, 999])
print("Sai vì dùng or:", wrong_same_uses_or(x, y))
print("Đúng vì dùng and:", solve(x, y))
assert wrong_same_uses_or(x, y) is True and solve(x, y) is False

Sai vì dùng or: True
Đúng vì dùng and: False


## 13. Nhận diện dạng bài và vận dụng

Dấu hiệu: **hai cấu trúc cây có cùng vị trí tương ứng**. Mẫu chung: kiểm tra base case trước, so sánh node hiện tại, gọi lại trên cặp con trái và phải.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Thay một giá trị ở **lá sâu nhất** và dự đoán tại sao việc chỉ kiểm tra hai gốc sẽ bỏ sót sai khác.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [24]:
a=tree([1,2,3,4,5]);b=tree([1,2,3,4,999]);print('Giống nhau?',solve(a,b))

Giống nhau? False


### 14.1. Pattern nhận diện: “hai đối tượng đệ quy tương ứng”

Dấu hiệu của bài Same Tree:
- Có **hai** cấu trúc cây cần đối chiếu.
- Ở mỗi bước, cần xét **cùng vị trí** (trái-trái, phải-phải).
- Có trường hợp dừng `None`; có phép ghép kết quả từ cây con.

Khi gặp bài khác, ta có thể tái sử dụng cấu trúc **so sánh từng cặp**;
nhưng phải đổi tiêu chí cho đúng đề (ví dụ: so ảnh gương #101 sẽ so **trái-phải**, không còn trái-trái).

**Đừng học thuộc mỗi code.** Hãy viết ra “hai dữ liệu con nào tương ứng?” trước khi chọn recursion.

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

`t = tree([5,2,8])`; dự đoán chính xác `print(t.val, t.left.val, t.right.val)` và `print(t.left.left is None)`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [25]:
t = tree([5,2,8])
print(t.val, t.left.val, t.right.val)
print(t.left.left is None)

5 2 8
True


### Bài 2 — Trace bằng tay (hiểu bản chất)

So sánh `p = [1,2,3]` và `q = [1,2,4]`: vẽ thứ tự các cặp node được xét (bao gồm `None`) cho đến khi tìm ra điểm khác. Kết quả cuối là gì?

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_same_tree(p, q)` bằng **đệ quy**, không dùng `solve` hoặc `Solution`. Phân biệt ba tình huống: cùng rỗng, chỉ một rỗng, cả hai có giá trị.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [26]:
def exercise_same_tree(p, q):
    # TODO 1: nếu cả hai là None, kết luận gì?
    # TODO 2: nếu đúng một cái là None, kết luận gì?
    # TODO 3: so sánh giá trị tại gốc.
    # TODO 4: so sánh cả nhánh trái và nhánh phải.
    raise NotImplementedError

# Khi hoàn thiện: assert exercise_same_tree(tree([1,2]), tree([1,2]))

### Bài 4 — TODO: Chuyển giao sang biến thể

Không dùng đệ quy: viết `exercise_same_tree_stack(p,q)` bằng **stack** chứa các cặp node. Dùng `pop()` để lấy cặp đang xét. Mục tiêu là thấy đệ quy có thể chuyển thành vòng lặp.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [27]:
def exercise_same_tree_stack(p, q):
    # TODO: stack ban đầu chứa (p,q); lặp cho đến rỗng.
    # Nhớ: với mỗi cặp khác None, thêm (left,left), (right,right).
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_first_mismatch(p,q)` trả `None` nếu giống nhau, ngược lại trả đường đi đến **vị trí khác đầu tiên** theo thứ tự trái trước, phải sau: `"root"` nếu khác ở gốc, `"L"` ở con trái, `"RR"` ở phải-phải. Không cần so sánh sâu sau khi phát hiện khác.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [28]:
def exercise_first_mismatch(p, q):
    # TODO: duyệt DFS cùng lúc trên hai cây, mang theo path "", "L", "R".
    # Khi khác: trả path hoặc "root" nếu path rỗng.
    raise NotImplementedError

### Bài 6 — Tự viết hàm so **hình dáng**, bỏ qua giá trị nút

**Đề:** Viết `exercise_same_shape(p,q)` trả `True` nếu hai cây có đúng các vị trí nút giống nhau,
kể cả khi giá trị các nút khác nhau. Ví dụ `[1,2,3]` và `[8,9,10]` cùng hình dáng ⇒ `True`,
`[1,2]` và `[8,None,9]` ⇒ `False`.

**Gợi ý cấp 1:** giữ hai trường hợp `None`, bỏ điều kiện `a.val == b.val`.

<details><summary>Gợi ý cấp 2</summary>Đệ quy cả trái-trái và phải-phải, ghép bằng <code>and</code>.</details>

<details><summary>Gợi ý cấp 3</summary>Chỉ sau khi xác nhận cả hai đều không None, gọi lại hàm hai lần cho con trái và con phải.</details>

In [29]:
def exercise_same_shape(p, q):
    # TODO 1: cả hai rỗng => True.
    # TODO 2: chỉ một bên rỗng => False.
    # TODO 3: gọi đệ quy lên hai cặp con tương ứng.
    raise NotImplementedError

# Sau khi tự viết, hãy chạy:
# assert exercise_same_shape(tree([1,2,3]), tree([8,9,10]))
# assert not exercise_same_shape(tree([1,2]), tree([8,None,9]))

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Hai cây chỉ giống khi **giá trị và cấu trúc** đều khớp; không thể bỏ qua `None`.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Với stack, lưu `(node_p,node_q)`; với bài đường đi, thêm `path + "L"` hoặc `path + "R"`.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Đệ quy: `same(l1,l2) and same(r1,r2)`. Stack: lấy cặp, xử lý None, so giá trị, rồi thêm cặp con.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Dòng 1: `5 2 8`. Dòng 2: `True` vì node giá trị `2` không có con trái.

### Bài 2 — Trace tay

Gốc `(1,1)` khớp → cặp trái `(2,2)` khớp → `(None,None)` đúng hai lần → cặp phải `(3,4)` khác giá trị → kết quả **False**. Có thể bỏ qua cặp rỗng của node `3` vì phát hiện khác trước.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [30]:
def answer_same_tree(p, q):
    if p is None and q is None: return True
    if p is None or q is None: return False
    if p.val != q.val: return False
    return answer_same_tree(p.left,q.left) and answer_same_tree(p.right,q.right)

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [31]:
def answer_same_tree_stack(p,q):
    stack=[(p,q)]
    while stack:
        a,b=stack.pop()
        if a is None and b is None: continue
        if a is None or b is None or a.val!=b.val: return False
        stack.append((a.right,b.right))
        stack.append((a.left,b.left))
    return True

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [32]:
def answer_first_mismatch(p,q):
    def visit(a,b,path):
        if a is None and b is None: return None
        if a is None or b is None or a.val!=b.val: return path or "root"
        left=visit(a.left,b.left,path+"L")
        if left is not None: return left
        return visit(a.right,b.right,path+"R")
    return visit(p,q,"")

### Đáp án bài 6 — so cấu trúc, không so `val`

Khác biệt đúng **một** ý so với Same Tree: không hề quan tâm `val`.
Đây là cách luyện thói quen tách tiêu chí “hình dáng” và tiêu chí “giá trị”.

In [33]:
def answer_same_shape(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None:
        return False
    return (answer_same_shape(p.left, q.left)
            and answer_same_shape(p.right, q.right))

assert answer_same_shape(tree([1, 2, 3]), tree([8, 9, 10]))
assert not answer_same_shape(tree([1, 2]), tree([8, None, 9]))
assert answer_same_shape(None, None)
print("PASS bài 6 — so cấu trúc")

PASS bài 6 — so cấu trúc


### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [34]:
assert answer_same_tree(tree([1,2,3]),tree([1,2,3]))
assert not answer_same_tree(tree([1,2]),tree([1,None,2]))
assert answer_same_tree_stack(None,None)
assert not answer_same_tree_stack(tree([1]),None)
assert answer_first_mismatch(tree([1,2,3]),tree([1,8,3]))=="L"
assert answer_first_mismatch(tree([1]),tree([2]))=="root"
assert answer_first_mismatch(tree([1,2]),tree([1,2])) is None
print("PASS: đáp án 3 bài coding của phần luyện tập")

PASS: đáp án 3 bài coding của phần luyện tập


### Tự đánh giá sau khi học #100

Trước khi chuyển sang bài mới, hãy tự trả lời thành tiếng:

1. Vì sao `[1,2]` khác `[1,None,2]` dù có cùng hai số?
2. Vì sao phải kiểm tra `a is None` trước khi đọc `a.val`?
3. Khi nào `and` dừng sớm mà không so nhánh phải?
4. Stack đẩy cặp `(left,left)` hay `(left,right)`? Vì sao?
5. `n`, `m`, `h` trong Big-O khác nhau ở đâu?
6. Nếu yêu cầu chỉ so hình dáng, dòng điều kiện nào cần bỏ?

**Bước tự kiểm cuối:** mở một notebook mới, không nhìn code, viết lại `same(a,b)` từ 3 base cases, sau đó thử cây rỗng và cây chỉ khác một chiếc lá.

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Thay vì lưu cả cây dưới dạng chuỗi/tuple, ta so sánh **ngay khi gặp hai node**. Trường hợp dừng là `None`: cả hai None → giống; chỉ một None → khác.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.